In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"

hourly = pd.read_csv(PROCESSED_DIR / "hourly_pickups.csv", parse_dates=["hour"])
clusters = pd.read_csv(PROCESSED_DIR / "station_clusters.csv")
weather = pd.read_csv(PROCESSED_DIR / "weather_hourly.csv", parse_dates=["hour"])

print(f"Hourly pickups: {hourly.shape}")
print(f"Station clusters: {clusters.shape}")
print(f"Weather: {weather.shape}")

## aggregating to cluster level

In [ ]:
df = hourly.merge(clusters[["station_id", "cluster_id"]], on="station_id", how="inner")
cluster_agg = df.groupby(["cluster_id", "hour"])["pickups"].sum().reset_index()

cluster_cap = clusters.drop_duplicates("cluster_id")[["cluster_id", "cluster_capacity"]]
cluster_agg = cluster_agg.merge(cluster_cap, on="cluster_id")

cluster_agg = cluster_agg.sort_values(["cluster_id", "hour"])
cluster_agg["lag_24h"] = cluster_agg.groupby("cluster_id")["pickups"].shift(24)  # same hour yesterday
cluster_agg = cluster_agg.dropna(subset=["lag_24h"])  # drops first 24h per cluster

print(f"Cluster-hour rows: {len(cluster_agg):,}")
print(f"Clusters: {cluster_agg['cluster_id'].nunique()}, Hours: {cluster_agg['hour'].nunique()}")
print(f"Date range: {cluster_agg['hour'].min()} to {cluster_agg['hour'].max()}")

## merging weather

In [ ]:
cluster_agg = cluster_agg.merge(weather, on="hour", how="left")
print(f"After weather merge: {cluster_agg.shape}")
print(f"Weather nulls: {cluster_agg[['temperature_c','precipitation_mm','wind_speed_kmh']].isnull().sum().to_dict()}")

## temporal and weather features

In [ ]:
HOLIDAYS = {
    "2025-01-01", "2025-02-17", "2025-04-18", "2025-05-19",
    "2025-07-01", "2025-08-04", "2025-09-01", "2025-10-13",
    "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-02-16",
}

df = cluster_agg.copy()
df["hour_of_day"] = df["hour"].dt.hour
df["day_of_week"] = df["hour"].dt.dayofweek  # 0=Monday
df["month"] = df["hour"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_rush_hour"] = (
    (df["hour_of_day"].between(7, 9) | df["hour_of_day"].between(16, 18)) &
    (df["is_weekend"] == 0)
).astype(int)
df["is_holiday"] = df["hour"].dt.date.astype(str).isin(HOLIDAYS).astype(int)
df["is_rainy"] = (df["precipitation_mm"] > 0).astype(int)

FEATURES = [
    "cluster_capacity",
    "hour_of_day", "day_of_week", "month", "is_weekend", "is_rush_hour", "is_holiday",
    "temperature_c", "precipitation_mm", "wind_speed_kmh", "is_rainy",
    "lag_24h",
]
print(f"Features ({len(FEATURES)}): {FEATURES}")
print(df[FEATURES + ["pickups"]].describe().round(2))

## train / val / test split

In [ ]:
train = df[df["hour"].dt.year == 2025].copy()
val   = df[(df["hour"].dt.year == 2026) & (df["hour"].dt.month == 1)].copy()
test  = df[(df["hour"].dt.year == 2026) & (df["hour"].dt.month == 2)].copy()

print(f"Train: {train.shape}  ({train['hour'].min().date()} to {train['hour'].max().date()})")
print(f"Val:   {val.shape}  ({val['hour'].min().date()} to {val['hour'].max().date()})")
print(f"Test:  {test.shape}  ({test['hour'].min().date()} to {test['hour'].max().date()})")
print(f"\nTarget stats:")
print(f"  train  mean={train['pickups'].mean():.1f}  std={train['pickups'].std():.1f}")
print(f"  val    mean={val['pickups'].mean():.1f}  std={val['pickups'].std():.1f}")
print(f"  test   mean={test['pickups'].mean():.1f}  std={test['pickups'].std():.1f}")

## saving

In [ ]:
KEEP_COLS = ["cluster_id", "hour", "pickups"] + FEATURES

train[KEEP_COLS].to_csv(PROCESSED_DIR / "train.csv", index=False)
val[KEEP_COLS].to_csv(PROCESSED_DIR / "val.csv", index=False)
test[KEEP_COLS].to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Saved train.csv, val.csv, test.csv to data/processed/")
print(f"Columns ({len(KEEP_COLS)}): {KEEP_COLS}")